In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [3]:
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
print("Model loaded.")

Model loaded.


In [4]:
arabic_text = "اسمي أحمد وأنا من سوريا"  # "My name is Ahmad and I am from Syria"

# NLLB needs source and target language codes
tokenizer.src_lang = "arb_Arab"  # Modern Standard Arabic
inputs = tokenizer(arabic_text, return_tensors="pt").to(device)

translated_tokens = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("deu_Latn"),  # German
    max_length=128,
)

german_translation = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
print(f"Arabic:  {arabic_text}")
print(f"German:  {german_translation}")

Arabic:  اسمي أحمد وأنا من سوريا
German:  Ich bin Ahmed und ich komme aus Syrien.


In [5]:
test_sentences = [
    # 1. Simple greeting (you've seen one already)
    "مرحبا، كيف حالك؟",
    # 2. Personal information (typical police context)
    "اسمي محمد علي، ولدت في حلب عام 1987",
    # 3. An address (proper nouns + numbers)
    "أسكن في شارع الرشيد رقم 42، دمشق",
    # 4. A document type (technical vocabulary)
    "هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية",
    # 5. A date with religious calendar reference
    "تاريخ الإصدار: 15 رمضان 1444 هجري",
    # 6. Numbers and identifiers (high stakes for police)
    "رقم الهوية الشخصية: 0123456789",
    # 7. A medical term
    "المريض يعاني من ارتفاع ضغط الدم وداء السكري",
    # 8. A legal phrase
    "بناءً على المادة الثانية من قانون العقوبات",
    # 9. A common name with a transliteration challenge
    "السيد عبد الرحمن الحسيني",
    # 10. An expression with cultural context
    "إن شاء الله سأصل غدا في المساء",
]

for i, ar_text in enumerate(test_sentences, 1):
    tokenizer.src_lang = "arb_Arab"
    inputs = tokenizer(ar_text, return_tensors="pt").to(device)
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("deu_Latn"),
        max_length=128,
    )
    de_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    print(f"{i}. AR: {ar_text}")
    print(f"   DE: {de_text}")
    print()

1. AR: مرحبا، كيف حالك؟
   DE: Hey, wie geht's Ihnen?

2. AR: اسمي محمد علي، ولدت في حلب عام 1987
   DE: Mein Name ist Mohammed Ali, ich wurde 1987 in Aleppo geboren.

3. AR: أسكن في شارع الرشيد رقم 42، دمشق
   DE: Ich wohne in der 42 R.R.R.D.M.

4. AR: هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية
   DE: Das ist ein offizieller Geburtsurkunde des Innenministeriums.

5. AR: تاريخ الإصدار: 15 رمضان 1444 هجري
   DE: Veröffentlichung: 15. Ramadan 1444 Hijri

6. AR: رقم الهوية الشخصية: 0123456789
   DE: Ihre persönliche Personalausweisnummer: 0123456789

7. AR: المريض يعاني من ارتفاع ضغط الدم وداء السكري
   DE: Der Patient hat Bluthochdruck und Diabetes.

8. AR: بناءً على المادة الثانية من قانون العقوبات
   DE: Gemäß Artikel 2 des Strafgesetzbuches

9. AR: السيد عبد الرحمن الحسيني
   DE: Herr Abdel Rahman Al-Husseini .

10. AR: إن شاء الله سأصل غدا في المساء
   DE: Ich komme morgen Abend.



In [6]:
import sacrebleu

# Three sentences with reference translations (from a fluent speaker — me, in this case)
references_for_bleu = [
    {
        "ar": "اسمي أحمد وأنا من سوريا",
        "ref_de": "Ich heiße Ahmed und ich komme aus Syrien.",
    },
    {
        "ar": "أسكن في شارع الرشيد رقم 42، دمشق",
        "ref_de": "Ich wohne in der Rashid-Straße Nummer 42 in Damaskus.",
    },
    {
        "ar": "رقم الهوية الشخصية: 0123456789",
        "ref_de": "Personalausweisnummer: 0123456789",
    },
]

hypotheses = []
references = []

for item in references_for_bleu:
    tokenizer.src_lang = "arb_Arab"
    inputs = tokenizer(item["ar"], return_tensors="pt").to(device)
    out = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("deu_Latn"),
        max_length=128,
    )
    de_translation = tokenizer.decode(out[0], skip_special_tokens=True)
    hypotheses.append(de_translation)
    references.append(item["ref_de"])
    print(f"AR:  {item['ar']}")
    print(f"REF: {item['ref_de']}")
    print(f"HYP: {de_translation}")
    print()

# sacrebleu expects: hypotheses as a list, references as a list of lists
# (because there can be multiple acceptable references)
bleu = sacrebleu.corpus_bleu(hypotheses, [references])
print(f"BLEU: {bleu.score:.2f}")

AR:  اسمي أحمد وأنا من سوريا
REF: Ich heiße Ahmed und ich komme aus Syrien.
HYP: Ich bin Ahmed und ich komme aus Syrien.

AR:  أسكن في شارع الرشيد رقم 42، دمشق
REF: Ich wohne in der Rashid-Straße Nummer 42 in Damaskus.
HYP: Ich wohne in der 42 R.R.R.D.M.

AR:  رقم الهوية الشخصية: 0123456789
REF: Personalausweisnummer: 0123456789
HYP: Ihre persönliche Personalausweisnummer: 0123456789

BLEU: 38.32


In [7]:
test_sentences = [
    # 1. Simple greeting (you've seen one already)
    "مرحبا، كيف حالك؟",
    # 2. Personal information (typical police context)
    "اسمي محمد علي، ولدت في حلب عام 1987",
    # 3. An address (proper nouns + numbers)
    "أسكن في شارع الرشيد رقم 42، دمشق",
    # 4. A document type (technical vocabulary)
    "هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية",
    # 5. A date with religious calendar reference
    "تاريخ الإصدار: 15 رمضان 1444 هجري",
    # 6. Numbers and identifiers (high stakes for police)
    "رقم الهوية الشخصية: 0123456789",
    # 7. A medical term
    "المريض يعاني من ارتفاع ضغط الدم وداء السكري",
    # 8. A legal phrase
    "بناءً على المادة الثانية من قانون العقوبات",
    # 9. A common name with a transliteration challenge
    "السيد عبد الرحمن الحسيني",
    # 10. An expression with cultural context
    "إن شاء الله سأصل غدا في المساء",
]

for i, ar_text in enumerate(test_sentences, 1):
    tokenizer.src_lang = "arb_Arab"
    inputs = tokenizer(ar_text, return_tensors="pt").to(device)
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("deu_Latn"),
        max_length=128,
    )
    de_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    print(f"{i}. AR: {ar_text}")
    print(f"   DE: {de_text}")
    print()

1. AR: مرحبا، كيف حالك؟
   DE: Hey, wie geht's Ihnen?

2. AR: اسمي محمد علي، ولدت في حلب عام 1987
   DE: Mein Name ist Mohammed Ali, ich wurde 1987 in Aleppo geboren.

3. AR: أسكن في شارع الرشيد رقم 42، دمشق
   DE: Ich wohne in der 42 R.R.R.D.M.

4. AR: هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية
   DE: Das ist ein offizieller Geburtsurkunde des Innenministeriums.

5. AR: تاريخ الإصدار: 15 رمضان 1444 هجري
   DE: Veröffentlichung: 15. Ramadan 1444 Hijri

6. AR: رقم الهوية الشخصية: 0123456789
   DE: Ihre persönliche Personalausweisnummer: 0123456789

7. AR: المريض يعاني من ارتفاع ضغط الدم وداء السكري
   DE: Der Patient hat Bluthochdruck und Diabetes.

8. AR: بناءً على المادة الثانية من قانون العقوبات
   DE: Gemäß Artikel 2 des Strafgesetzbuches

9. AR: السيد عبد الرحمن الحسيني
   DE: Herr Abdel Rahman Al-Husseini .

10. AR: إن شاء الله سأصل غدا في المساء
   DE: Ich komme morgen Abend.



In [8]:
test_sentences = [
    # 1. Simple greeting (you've seen one already)
    "مرحبا، كيف حالك؟",
    # 2. Personal information (typical police context)
    "اسمي محمد علي، ولدت في حلب عام 1987",
    # 3. An address (proper nouns + numbers)
    "أسكن في شارع الرشيد رقم 42، دمشق",
    # 4. A document type (technical vocabulary)
    "هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية",
    # 5. A date with religious calendar reference
    "تاريخ الإصدار: 15 رمضان 1444 هجري",
    # 6. Numbers and identifiers (high stakes for police)
    "رقم الهوية الشخصية: 0123456789",
    # 7. A medical term
    "المريض يعاني من ارتفاع ضغط الدم وداء السكري",
    # 8. A legal phrase
    "بناءً على المادة الثانية من قانون العقوبات",
    # 9. A common name with a transliteration challenge
    "السيد عبد الرحمن الحسيني",
    # 10. An expression with cultural context
    "إن شاء الله سأصل غدا في المساء",
]

for i, ar_text in enumerate(test_sentences, 1):
    tokenizer.src_lang = "arb_Arab"
    inputs = tokenizer(ar_text, return_tensors="pt").to(device)
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("deu_Latn"),
        max_length=128,
    )
    de_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    print(f"{i}. AR: {ar_text}")
    print(f"   DE: {de_text}")
    print()

1. AR: مرحبا، كيف حالك؟
   DE: Hey, wie geht's Ihnen?

2. AR: اسمي محمد علي، ولدت في حلب عام 1987
   DE: Mein Name ist Mohammed Ali, ich wurde 1987 in Aleppo geboren.

3. AR: أسكن في شارع الرشيد رقم 42، دمشق
   DE: Ich wohne in der 42 R.R.R.D.M.

4. AR: هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية
   DE: Das ist ein offizieller Geburtsurkunde des Innenministeriums.

5. AR: تاريخ الإصدار: 15 رمضان 1444 هجري
   DE: Veröffentlichung: 15. Ramadan 1444 Hijri

6. AR: رقم الهوية الشخصية: 0123456789
   DE: Ihre persönliche Personalausweisnummer: 0123456789

7. AR: المريض يعاني من ارتفاع ضغط الدم وداء السكري
   DE: Der Patient hat Bluthochdruck und Diabetes.

8. AR: بناءً على المادة الثانية من قانون العقوبات
   DE: Gemäß Artikel 2 des Strafgesetzbuches

9. AR: السيد عبد الرحمن الحسيني
   DE: Herr Abdel Rahman Al-Husseini .

10. AR: إن شاء الله سأصل غدا في المساء
   DE: Ich komme morgen Abend.



In [9]:
test_sentences = [
    # 1. Simple greeting (you've seen one already)
    "مرحبا، كيف حالك؟",
    # 2. Personal information (typical police context)
    "اسمي محمد علي، ولدت في حلب عام 1987",
    # 3. An address (proper nouns + numbers)
    "أسكن في شارع الرشيد رقم 42، دمشق",
    # 4. A document type (technical vocabulary)
    "هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية",
    # 5. A date with religious calendar reference
    "تاريخ الإصدار: 15 رمضان 1444 هجري",
    # 6. Numbers and identifiers (high stakes for police)
    "رقم الهوية الشخصية: 0123456789",
    # 7. A medical term
    "المريض يعاني من ارتفاع ضغط الدم وداء السكري",
    # 8. A legal phrase
    "بناءً على المادة الثانية من قانون العقوبات",
    # 9. A common name with a transliteration challenge
    "السيد عبد الرحمن الحسيني",
    # 10. An expression with cultural context
    "إن شاء الله سأصل غدا في المساء",
]

for i, ar_text in enumerate(test_sentences, 1):
    tokenizer.src_lang = "arb_Arab"
    inputs = tokenizer(ar_text, return_tensors="pt").to(device)
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("deu_Latn"),
        max_length=128,
    )
    de_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    print(f"{i}. AR: {ar_text}")
    print(f"   DE: {de_text}")
    print()

1. AR: مرحبا، كيف حالك؟
   DE: Hey, wie geht's Ihnen?

2. AR: اسمي محمد علي، ولدت في حلب عام 1987
   DE: Mein Name ist Mohammed Ali, ich wurde 1987 in Aleppo geboren.

3. AR: أسكن في شارع الرشيد رقم 42، دمشق
   DE: Ich wohne in der 42 R.R.R.D.M.

4. AR: هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية
   DE: Das ist ein offizieller Geburtsurkunde des Innenministeriums.

5. AR: تاريخ الإصدار: 15 رمضان 1444 هجري
   DE: Veröffentlichung: 15. Ramadan 1444 Hijri

6. AR: رقم الهوية الشخصية: 0123456789
   DE: Ihre persönliche Personalausweisnummer: 0123456789

7. AR: المريض يعاني من ارتفاع ضغط الدم وداء السكري
   DE: Der Patient hat Bluthochdruck und Diabetes.

8. AR: بناءً على المادة الثانية من قانون العقوبات
   DE: Gemäß Artikel 2 des Strafgesetzbuches

9. AR: السيد عبد الرحمن الحسيني
   DE: Herr Abdel Rahman Al-Husseini .

10. AR: إن شاء الله سأصل غدا في المساء
   DE: Ich komme morgen Abend.



In [10]:
test_sentences = [
    # 1. Simple greeting (you've seen one already)
    "مرحبا، كيف حالك؟",
    # 2. Personal information (typical police context)
    "اسمي محمد علي، ولدت في حلب عام 1987",
    # 3. An address (proper nouns + numbers)
    "أسكن في شارع الرشيد رقم 42، دمشق",
    # 4. A document type (technical vocabulary)
    "هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية",
    # 5. A date with religious calendar reference
    "تاريخ الإصدار: 15 رمضان 1444 هجري",
    # 6. Numbers and identifiers (high stakes for police)
    "رقم الهوية الشخصية: 0123456789",
    # 7. A medical term
    "المريض يعاني من ارتفاع ضغط الدم وداء السكري",
    # 8. A legal phrase
    "بناءً على المادة الثانية من قانون العقوبات",
    # 9. A common name with a transliteration challenge
    "السيد عبد الرحمن الحسيني",
    # 10. An expression with cultural context
    "إن شاء الله سأصل غدا في المساء",
]

for i, ar_text in enumerate(test_sentences, 1):
    tokenizer.src_lang = "arb_Arab"
    inputs = tokenizer(ar_text, return_tensors="pt").to(device)
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("deu_Latn"),
        max_length=128,
    )
    de_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    print(f"{i}. AR: {ar_text}")
    print(f"   DE: {de_text}")
    print()

1. AR: مرحبا، كيف حالك؟
   DE: Hey, wie geht's Ihnen?

2. AR: اسمي محمد علي، ولدت في حلب عام 1987
   DE: Mein Name ist Mohammed Ali, ich wurde 1987 in Aleppo geboren.

3. AR: أسكن في شارع الرشيد رقم 42، دمشق
   DE: Ich wohne in der 42 R.R.R.D.M.

4. AR: هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية
   DE: Das ist ein offizieller Geburtsurkunde des Innenministeriums.

5. AR: تاريخ الإصدار: 15 رمضان 1444 هجري
   DE: Veröffentlichung: 15. Ramadan 1444 Hijri

6. AR: رقم الهوية الشخصية: 0123456789
   DE: Ihre persönliche Personalausweisnummer: 0123456789

7. AR: المريض يعاني من ارتفاع ضغط الدم وداء السكري
   DE: Der Patient hat Bluthochdruck und Diabetes.

8. AR: بناءً على المادة الثانية من قانون العقوبات
   DE: Gemäß Artikel 2 des Strafgesetzbuches

9. AR: السيد عبد الرحمن الحسيني
   DE: Herr Abdel Rahman Al-Husseini .

10. AR: إن شاء الله سأصل غدا في المساء
   DE: Ich komme morgen Abend.



In [11]:
test_sentences = [
    # 1. Simple greeting (you've seen one already)
    "مرحبا، كيف حالك؟",
    # 2. Personal information (typical police context)
    "اسمي محمد علي، ولدت في حلب عام 1987",
    # 3. An address (proper nouns + numbers)
    "أسكن في شارع الرشيد رقم 42، دمشق",
    # 4. A document type (technical vocabulary)
    "هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية",
    # 5. A date with religious calendar reference
    "تاريخ الإصدار: 15 رمضان 1444 هجري",
    # 6. Numbers and identifiers (high stakes for police)
    "رقم الهوية الشخصية: 0123456789",
    # 7. A medical term
    "المريض يعاني من ارتفاع ضغط الدم وداء السكري",
    # 8. A legal phrase
    "بناءً على المادة الثانية من قانون العقوبات",
    # 9. A common name with a transliteration challenge
    "السيد عبد الرحمن الحسيني",
    # 10. An expression with cultural context
    "إن شاء الله سأصل غدا في المساء",
]

for i, ar_text in enumerate(test_sentences, 1):
    tokenizer.src_lang = "arb_Arab"
    inputs = tokenizer(ar_text, return_tensors="pt").to(device)
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("deu_Latn"),
        max_length=128,
    )
    de_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    print(f"{i}. AR: {ar_text}")
    print(f"   DE: {de_text}")
    print()

1. AR: مرحبا، كيف حالك؟
   DE: Hey, wie geht's Ihnen?

2. AR: اسمي محمد علي، ولدت في حلب عام 1987
   DE: Mein Name ist Mohammed Ali, ich wurde 1987 in Aleppo geboren.

3. AR: أسكن في شارع الرشيد رقم 42، دمشق
   DE: Ich wohne in der 42 R.R.R.D.M.

4. AR: هذه شهادة ميلاد رسمية صادرة من وزارة الداخلية
   DE: Das ist ein offizieller Geburtsurkunde des Innenministeriums.

5. AR: تاريخ الإصدار: 15 رمضان 1444 هجري
   DE: Veröffentlichung: 15. Ramadan 1444 Hijri

6. AR: رقم الهوية الشخصية: 0123456789
   DE: Ihre persönliche Personalausweisnummer: 0123456789

7. AR: المريض يعاني من ارتفاع ضغط الدم وداء السكري
   DE: Der Patient hat Bluthochdruck und Diabetes.

8. AR: بناءً على المادة الثانية من قانون العقوبات
   DE: Gemäß Artikel 2 des Strafgesetzbuches

9. AR: السيد عبد الرحمن الحسيني
   DE: Herr Abdel Rahman Al-Husseini .

10. AR: إن شاء الله سأصل غدا في المساء
   DE: Ich komme morgen Abend.

